# Class 2 — Embeddings & Vector Databases

**Week 6: Foundations of RAG and Chatbots**

### Learning objectives
By the end of this notebook you will be able to:
- Explain what an embedding is and why similar meanings land near each other in vector space
- Compute cosine similarity between two vectors by hand and with NumPy
- Generate real embeddings with a local model and rank a small set of documents against a query
- Describe, at a high level, what a vector database (Chroma, FAISS, Pinecone) adds beyond a plain list

> This notebook contains a fully runnable demo: real embeddings, real cosine similarity, a real nearest-neighbor
> match. Embeddings run locally — no API key required for this class.

## Setup

Install `sentence-transformers` once. Embeddings in this class run **locally** with `all-MiniLM-L6-v2` —
a small model that is plenty for ranking a handful of sentences. No API key is required.

Class 3 still uses Groq for generation (`llama-3.3-70b-versatile`); add a Colab secret named `GROQ_API_KEY`
before that notebook.

In [7]:
!pip install -q sentence-transformers

from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "all-MiniLM-L6-v2"
_embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f"Loaded local embedding model: {EMBEDDING_MODEL}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loaded local embedding model: all-MiniLM-L6-v2


## Concept 1 — What Is an Embedding, Really?

An embedding is a list of numbers (a vector) that represents the *meaning* of a piece of text. A trained embedding
model turns text into this vector. Two texts with similar meaning end up with vectors that are close together in
that numeric space — "dog" and "puppy" land near each other, while "dog" and "stock market" land far apart.

Real embedding models produce vectors with hundreds or thousands of numbers (dimensions). We can't draw that, but
the geometry idea is identical to the small examples below.

## Concept 2 — Measuring Similarity With Cosine

Before we load a real embedding model, let's build intuition with tiny, fake 2-dimensional "embeddings" we can plot mentally.
Cosine similarity measures the angle between two vectors — it asks "do they point in the same direction?" — and
ignores their length. Scores run roughly from -1 (opposite) to 1 (identical direction).

In [2]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a, dtype=float)
    b = np.array(b, dtype=float)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Tiny made-up 2D "embeddings" just to build intuition -- real embeddings have hundreds of dimensions.
dog    = [0.9, 0.2]
puppy  = [0.85, 0.3]
stocks = [-0.1, 0.95]

print("dog vs puppy: ", cosine_similarity(dog, puppy))
print("dog vs stocks:", cosine_similarity(dog, stocks))

dog vs puppy:  0.9927337820337083
dog vs stocks: 0.11354659116073189


Notice "dog" and "puppy" score much higher (closer to 1) than "dog" and "stocks" — exactly what we'd hope for.
Now let's do this with **real** embeddings from a local model instead of numbers we made up.

## Concept 3 — Generate Real Embeddings and Rank Documents

We'll embed a small set of documents and one query using a local sentence-transformers model (`all-MiniLM-L6-v2`),
then rank the documents by cosine similarity to the query. The highest-scoring document is the nearest neighbor —
the one we'd retrieve in a real RAG system.

In [3]:
def get_embedding(text, model=EMBEDDING_MODEL):
    """Return the embedding vector (a list of floats) for a piece of text."""
    return _embedder.encode(text).tolist()

documents = [
    "Password reset instructions for the employee portal: click 'Forgot password' and check your email.",
    "Our refund policy allows returns within 30 days of purchase with a valid receipt.",
    "Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.",
]

query = "How do I reset my password?"

query_vec = get_embedding(query)
doc_vecs = [get_embedding(doc) for doc in documents]
scored = sorted(
    zip(documents, doc_vecs),
    key=lambda pair: cosine_similarity(query_vec, pair[1]),
    reverse=True,
)
for rank, (doc, vec) in enumerate(scored, start=1):
    score = cosine_similarity(query_vec, vec)
    print(f"{rank}. [{score:.3f}] {doc}")

1. [0.653] Password reset instructions for the employee portal: click 'Forgot password' and check your email.
2. [0.032] Our refund policy allows returns within 30 days of purchase with a valid receipt.
3. [-0.015] Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.


The top-ranked document should be the password-reset one, even though the query never uses the exact word
"reset" the way the document text might. The embedding model matched **meaning**, not literal words — this is the
core trick behind semantic search.

## Concept 4 — Semantic Search vs. Keyword Search

- **Keyword search** matches exact words. A search for "puppy food" can miss an article titled "canine nutrition"
  even though it's exactly what the user wants.
- **Semantic search** (what we just did above) matches meaning. It handles synonyms, paraphrasing, and typos far
  more gracefully.
- Many production systems combine both — "hybrid search" — to get literal-match precision plus semantic recall.

Run the cell below on the same three documents. We'll use a query that *means* "password reset" but avoids those
words, then rank once by shared keywords and once by cosine similarity.

In [4]:
import re

STOP = {
    "a", "an", "the", "to", "of", "in", "on", "for", "and", "or", "if",
    "i", "my", "do", "how", "can", "into", "with", "is", "are", "it",
}

def keywords(text):
    words = re.findall(r"[a-z]+", text.lower())
    return {w for w in words if w not in STOP}

def keyword_overlap(query, doc):
    shared = keywords(query) & keywords(doc)
    return len(shared), shared

# Same meaning, almost no shared words — the slide's "puppy food" vs "canine nutrition" example.
pair_query = "puppy food"
pair_doc = "A short guide to canine nutrition for young dogs"
n, shared = keyword_overlap(pair_query, pair_doc)
print("SLIDE EXAMPLE")
print(f"  query: {pair_query!r}")
print(f"  doc:   {pair_doc!r}")
print(f"  keyword overlap: {n} {shared or set()}")
print(f"  cosine similarity: {cosine_similarity(get_embedding(pair_query), get_embedding(pair_doc)):.3f}")

# Reuse Concept 3's documents, but ask without saying "reset" or "password".
query = "I need help signing into my account"
print(f"\nQUERY: {query}\n")

print("Keyword ranking (shared content words):")
for i, doc in enumerate(sorted(documents, key=lambda d: keyword_overlap(query, d)[0], reverse=True), 1):
    n, shared = keyword_overlap(query, doc)
    print(f"  {i}. [{n} overlap {shared or set()}] {doc}")

print("\nSemantic ranking (cosine on embeddings):")
qvec = get_embedding(query)
for i, doc in enumerate(sorted(documents, key=lambda d: cosine_similarity(qvec, get_embedding(d)), reverse=True), 1):
    print(f"  {i}. [{cosine_similarity(qvec, get_embedding(doc)):.3f}] {doc}")

SLIDE EXAMPLE
  query: 'puppy food'
  doc:   'A short guide to canine nutrition for young dogs'
  keyword overlap: 0 set()
  cosine similarity: 0.635

QUERY: I need help signing into my account

Keyword ranking (shared content words):
  1. [0 overlap set()] Password reset instructions for the employee portal: click 'Forgot password' and check your email.
  2. [0 overlap set()] Our refund policy allows returns within 30 days of purchase with a valid receipt.
  3. [0 overlap set()] Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.

Semantic ranking (cosine on embeddings):
  1. [0.447] Password reset instructions for the employee portal: click 'Forgot password' and check your email.
  2. [0.040] Our refund policy allows returns within 30 days of purchase with a valid receipt.
  3. [-0.054] Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.


Keyword search misses the password-reset document because the query said "signing into" instead of "reset" or
"password." Semantic search still ranks it first — same meaning, different words. Hybrid search in production
often runs both and merges the rankings.

## Concept 5 — Vector Databases: Where Embeddings Live

For three documents, a Python list is plenty. For a million documents, you need a system built to find nearest
neighbors fast. A few common options:

- **Chroma** — open-source, runs locally or embedded in your app; great for prototyping.
- **FAISS** — Meta's similarity-search library; in-memory and extremely fast; no server, you manage persistence.
- **Pinecone** — fully-managed cloud vector database; scales to billions of vectors; ops handled for you.

All three do the same core job as the `sorted(...)` call above: store vectors, and find the nearest ones to a
query vector, just at a much larger scale. The tiny class below is that job with no extra library — then we
time a brute-force scan to see why an index starts to matter.

In [5]:
class MiniVectorStore:
    """The same job a vector database does, for a handful of documents."""

    def __init__(self):
        self.items = []

    def add(self, texts):
        for text in texts:
            self.items.append({"text": text, "embedding": get_embedding(text)})

    def query(self, text, n_results=2):
        q = get_embedding(text)
        ranked = sorted(
            self.items,
            key=lambda item: cosine_similarity(q, item["embedding"]),
            reverse=True,
        )
        return ranked[:n_results]


store = MiniVectorStore()
store.add(documents)

print("store.query('How do I reset my password?', n_results=2)")
for i, hit in enumerate(store.query("How do I reset my password?"), 1):
    print(f"  {i}. {hit['text']}")

print(
    "\nThat query() method is the same idea as Chroma's collection.query(), "
    "FAISS's index.search(), and Pinecone's index.query(): store vectors, return nearest neighbors.\n"
)

import time

print("Brute-force cosine over random vectors (MiniLM uses 384 dimensions):")
rng = np.random.default_rng(0)
dims = 384
query_vec = rng.normal(size=dims)
for n in (2_000, 20_000, 80_000):
    docs = rng.normal(size=(n, dims))
    t0 = time.perf_counter()
    scores = (docs @ query_vec) / (np.linalg.norm(docs, axis=1) * np.linalg.norm(query_vec))
    _ = np.argpartition(scores, -5)[-5:]
    elapsed_ms = (time.perf_counter() - t0) * 1000
    print(f"  {n:>6} vectors   {elapsed_ms:6.1f} ms")

print(
    "\nAt millions of vectors this linear scan gets expensive — that's when you pick "
    "Chroma (prototype), FAISS (fast local index), or Pinecone (managed cloud)."
)

store.query('How do I reset my password?', n_results=2)
  1. Password reset instructions for the employee portal: click 'Forgot password' and check your email.
  2. Our refund policy allows returns within 30 days of purchase with a valid receipt.

That query() method is the same idea as Chroma's collection.query(), FAISS's index.search(), and Pinecone's index.query(): store vectors, return nearest neighbors.

Brute-force cosine over random vectors (MiniLM uses 384 dimensions):
    2000 vectors     49.3 ms
   20000 vectors     98.0 ms
   80000 vectors    216.4 ms

At millions of vectors this linear scan gets expensive — that's when you pick Chroma (prototype), FAISS (fast local index), or Pinecone (managed cloud).


You do not need to install Chroma, FAISS, or Pinecone for this class. The wrapper above is enough to see the
API shape; Class 3 will keep using a plain Python list as the "vector store" so the RAG pipeline stays easy to
trace. Challenge 5 at the bottom asks you to pick one of the three for a real project and defend the tradeoff.

## Challenges

Each of these builds directly on the `get_embedding` and `cosine_similarity` functions above.

### Challenge 1 — Find the Odd One Out

Write five short sentences of your own, where four share a topic and one is unrelated. Embed all five, compute the
average pairwise similarity of each sentence to the other four, and print which one is the "odd one out."

**Acceptance criteria:** your code correctly identifies the unrelated sentence using similarity scores, not by
just eyeballing it.

In [ ]:
# TODO: write 5 sentences (4 related, 1 unrelated), embed them, and find the odd one out


In [8]:
sentences = [
    "Employees can work remotely two days per week.",
    "Remote employees must attend online team meetings.",
    "The company provides equipment for employees working remotely.",
    "Employees working from home should remain available during work hours.",
    "The company cafeteria serves fresh sandwiches every afternoon."
]

# Embed the sentences
embeddings = _embedder.encode(sentences)

# Find the sentence that is least similar to the others
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

similarity_matrix = cosine_similarity(embeddings)

# Calculate the average similarity of each sentence to all the others
average_similarity = similarity_matrix.mean(axis=1)

# The lowest average similarity is the odd one out
odd_index = np.argmin(average_similarity)

print("Odd one out:")
print(sentences[odd_index])

Odd one out:
The company cafeteria serves fresh sandwiches every afternoon.


### Challenge 2 — Swap the Query

Using the same `documents` list from Concept 3, write a new query that should match the refund-policy document
instead, and print the ranked results.

**Acceptance criteria:** the refund-policy document ranks first for your new query.

In [ ]:
# TODO: write a new query and print the re-ranked documents


In [11]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def rank_documents(query, documents):
    # Reshape the query embedding to a 2D array (1 sample, N features)
    query_vec = np.array(get_embedding(query)).reshape(1, -1)
    # Reshape each document embedding to a 2D array
    doc_embeddings = [np.array(get_embedding(doc)).reshape(1, -1) for doc in documents]

    scored_documents = []
    for doc, doc_vec in zip(documents, doc_embeddings):
        # sklearn's cosine_similarity returns a 2D array (e.g., [[score]]), so extract the scalar
        score = cosine_similarity(query_vec, doc_vec)[0][0]
        scored_documents.append((score, doc))

    ranked = sorted(scored_documents, key=lambda x: x[0], reverse=True)
    return ranked

query = "What is the company's policy for getting a refund after making a purchase?"

ranked = rank_documents(query, documents)

for score, doc in ranked:
    print(f"Score: {score:.4f} | {doc}")

Score: 0.7212 | Our refund policy allows returns within 30 days of purchase with a valid receipt.
Score: 0.1393 | Password reset instructions for the employee portal: click 'Forgot password' and check your email.
Score: 0.0144 | Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.


In [13]:
query = "What is the company's policy for getting a refund after making a purchase?"

# Assuming 'retrieve' was intended to rank documents, using the existing rank_documents function.
results = rank_documents(query, documents)

for score, doc in results:
    print(f"Score: {score:.4f} | {doc}")

Score: 0.7212 | Our refund policy allows returns within 30 days of purchase with a valid receipt.
Score: 0.1393 | Password reset instructions for the employee portal: click 'Forgot password' and check your email.
Score: 0.0144 | Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.


### Challenge 3 — Break It

Craft a query that you predict will fool cosine similarity into ranking an irrelevant document highly (for example,
a query that shares vocabulary with the wrong document but means something different). Run it and report whether
your prediction was correct.

**Acceptance criteria:** you state your prediction *before* running the code, then compare it to the actual
ranking.

In [ ]:
# TODO: craft a tricky query, predict the outcome in a comment, then run and compare


In [14]:
# TODO: craft a tricky query, predict the outcome in a comment, then run and compare

query = "I paid for something but now I don't want it. Can I get my money back?"

# Prediction: I expect the refund-policy document to rank first because
# "paid" and "money back" are related to getting a refund.

ranked = rank_documents(query, documents)

print("Query:", query)
print("\nRanked results:")

for score, doc in ranked:
    print(f"Score: {score:.4f} | {doc}")

# Compare the actual top result with the prediction
print("\nPrediction: Refund-policy document should rank first.")
print("Actual top result:", ranked[0][1])

Query: I paid for something but now I don't want it. Can I get my money back?

Ranked results:
Score: 0.4283 | Our refund policy allows returns within 30 days of purchase with a valid receipt.
Score: 0.1767 | Password reset instructions for the employee portal: click 'Forgot password' and check your email.
Score: -0.0730 | Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.

Prediction: Refund-policy document should rank first.
Actual top result: Our refund policy allows returns within 30 days of purchase with a valid receipt.


### Challenge 4 — Cosine vs. Euclidean

Write a `euclidean_distance(a, b)` function (straight-line distance between two vectors) and compare its ranking
of the `documents` list against `cosine_similarity`'s ranking for the same query. Are the rankings the same?

**Acceptance criteria:** you print both rankings side by side and state in a comment whether they agree.

In [ ]:
# TODO: implement euclidean_distance and compare its ranking to cosine_similarity's ranking


In [16]:
# TODO: implement euclidean_distance and compare its ranking
# to cosine_similarity's ranking

import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def euclidean_distance(a, b):
    return np.linalg.norm(np.array(a) - np.array(b))


# Create the query embedding
query_embedding = _embedder.encode(query)

# Get document embeddings
document_embeddings = _embedder.encode(documents)


# --- Cosine similarity ranking ---
cosine_scores = cosine_similarity(
    [query_embedding],
    document_embeddings
)[0]

cosine_ranking = np.argsort(cosine_scores)[::-1]


# --- Euclidean distance ranking ---
euclidean_scores = [
    euclidean_distance(query_embedding, doc_embedding)
    for doc_embedding in document_embeddings
]

# Smaller distance = more similar
euclidean_ranking = np.argsort(euclidean_scores)


# --- Print both rankings side by side ---
print("COSINE RANKING          EUCLIDEAN RANKING")
print("-" * 50)

for i in range(len(documents)):
    cosine_doc = documents[cosine_ranking[i]]
    euclidean_doc = documents[euclidean_ranking[i]]

    print(f"{i+1}. {cosine_doc}")
    print(f"   {i+1}. {euclidean_doc}")
    print()


# Check whether the rankings agree
if np.array_equal(cosine_ranking, euclidean_ranking):
    print("The rankings are the same.")
else:
    print("The rankings are different.")


# Comment:
# The rankings may be different because cosine similarity measures the
# angle between vectors, while Euclidean distance measures straight-line
# distance between them.

COSINE RANKING          EUCLIDEAN RANKING
--------------------------------------------------
1. Our refund policy allows returns within 30 days of purchase with a valid receipt.
   1. Our refund policy allows returns within 30 days of purchase with a valid receipt.

2. Password reset instructions for the employee portal: click 'Forgot password' and check your email.
   2. Password reset instructions for the employee portal: click 'Forgot password' and check your email.

3. Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.
   3. Office hours are 9am to 5pm, Monday through Friday, excluding public holidays.

The rankings are the same.


### Challenge 5 — Bonus: Make the Case for a Vector Database

In a comment or markdown-style triple-quoted string, argue in three sentences which of Chroma, FAISS, or Pinecone
you'd pick for a small internal support-chatbot project with under 5,000 documents, and why.

**Acceptance criteria:** your answer names one specific tradeoff (setup cost, scale, or ops burden) that drove
your choice.

In [ ]:
# TODO: argue for one vector database choice in 3 sentences, referencing a specific tradeoff


In [17]:
# I would choose Chroma for a small internal support chatbot with fewer than 5,000 documents.
# Its simple setup and local deployment make it easy to get started without the infrastructure
# and operational burden of a larger managed system like Pinecone, which is useful at greater scale.